# 10 — SRNet secondary robustness analysis

**Target:** *Signal Processing*.

This notebook adds a second independently trained neural steganalyzer after the frozen RDH experiment. The existing allocator, SRM-derived local-risk teacher, payloads, `alpha=0.25`, exact-recovery results, and enhanced residual-CNN endpoint are not changed.

Scientific status is deliberately conservative: the original test results are already known, therefore this experiment is a **post-hoc secondary robustness / external-validity analysis**, not a new pre-specified primary endpoint. The SRNet training/development protocol below is locked before SRNet scores the held-out test images. If the development sanity gate fails, the notebook stops before test scoring.

Architecture reference: M. Boroumand, M. Chen, J. Fridrich, “Deep Residual Network for Steganalysis of Digital Images,” *IEEE Transactions on Information Forensics and Security*, 14(5), 1181–1193, 2019, DOI: 10.1109/TIFS.2018.2871749.


In [ ]:
from pathlib import Path
import gc, hashlib, json, joblib, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdhlab.io import read_gray
from rdhlab.blockcodec import analyze_blocks
from rdhlab.pipeline import prepare_image_context, run_frozen_image_precomputed
from rdhlab.detectors import detector_metrics, paired_detector_bootstrap
from rdhlab.transfer import paired_method_bootstrap
from rdhlab.freeze_protocol import sha256_file, stable_id_hash
from rdhlab.srnet_secondary import (
    train_srnet, score_srnet, save_srnet, load_srnet,
    minimal_detection_error, paired_pe_bootstrap, parameter_count,
)

config=yaml.safe_load(Path('/workspace/config/experiment.yaml').read_text())
seed=int(config['project']['seed'])
fixed_fpr=float(config['detectors']['fixed_fpr'])
confidence=float(config['statistics']['confidence'])
n_boot=max(5000,int(config['statistics'].get('bootstrap_resamples',5000)))

manifest=pd.read_csv(config['dataset']['prepared_manifest'])
train=manifest[manifest.split=='train'].reset_index(drop=True)
test=manifest[manifest.split=='test'].reset_index(drop=True)
assert len(train)==6000 and len(test)==2000
assert train.source_id.astype(str).is_unique
assert test.source_id.astype(str).is_unique
assert set(train.source_id.astype(str)).isdisjoint(set(test.source_id.astype(str)))

allocator_path=Path('/workspace/config/frozen_allocator.json')
allocator=json.loads(allocator_path.read_text())
alpha=float(allocator['alpha'])
primary_bpp=float(allocator['teacher_payload_bpp'])
bs=int(allocator.get('block_size',config['dataset']['block_size']))
assert np.isclose(alpha,0.25)
assert np.isclose(primary_bpp,0.009)

out06=Path('/workspace/results/frozen_test_final')
complete=json.loads((out06/'test_run_complete.json').read_text())
protocol06=json.loads((out06/'test_protocol.json').read_text())
common=pd.read_csv(out06/'common_feasible_ids.csv')
common['source_id']=common.source_id.astype(str)
per06=pd.read_csv(out06/'per_image.csv')
per06['source_id']=per06.source_id.astype(str)

risk_path=Path('/workspace/results/models/srm_teacher_local_risk.joblib')
if sha256_file(risk_path)!=allocator['provenance_sha256']['srm_teacher_local_risk_joblib']:
    raise RuntimeError('Frozen SRM-derived local-risk model hash mismatch.')
if stable_id_hash(test.source_id.astype(str).tolist())!=complete['test_source_ids_sha256']:
    raise RuntimeError('Frozen test source-ID hash mismatch.')
local_risk=joblib.load(risk_path)

context_dir=out06/'contexts'/protocol06['context_tag']
if not context_dir.exists():
    raise RuntimeError(f'Frozen context cache missing: {context_dir}')

OUT=Path('/workspace/results/srnet_secondary')
MODEL_DIR=Path('/workspace/results/models')
OUT.mkdir(parents=True,exist_ok=True)
MODEL_DIR.mkdir(parents=True,exist_ok=True)

print('Frozen alpha:',alpha)
print('Primary payload:',primary_bpp)
print('Block size:',bs)
print('Train/test:',len(train),len(test))


## A. Freeze an SRNet-only development partition

The enhanced confirmatory CNN used TRAIN indices `0:2000` for fitting and `2000:2750` for detector development. SRNet uses a **disjoint** subset of the same TRAIN split: indices `2750:5250` for fitting and `5250:6000` for development. No TEST image is used here.

The SRNet is trained on stegos made by the deterministic `random` allocator at the already frozen primary rate `0.009 net bpp`. This avoids training directly on either method later compared (`joint` and `predictability`).


In [ ]:
SRNET_TRAIN_START=2750
SRNET_TRAIN_N=2500
SRNET_DEV_START=5250
SRNET_DEV_N=750
EPOCHS_STAGE1=10
EPOCHS_STAGE2=3
MICRO_PAIR_BATCH=2
ACCUMULATION_STEPS=4
LR_STAGE1=1e-3
LR_STAGE2=1e-4
WEIGHT_DECAY=2e-4
SANITY_AUC_THRESHOLD=0.65

srnet_train=train.iloc[SRNET_TRAIN_START:SRNET_TRAIN_START+SRNET_TRAIN_N].reset_index(drop=True)
srnet_dev=train.iloc[SRNET_DEV_START:SRNET_DEV_START+SRNET_DEV_N].reset_index(drop=True)
assert len(srnet_train)==SRNET_TRAIN_N and len(srnet_dev)==SRNET_DEV_N
assert set(srnet_train.source_id.astype(str)).isdisjoint(set(srnet_dev.source_id.astype(str)))
assert set(srnet_train.source_id.astype(str)).isdisjoint(set(test.source_id.astype(str)))
assert set(srnet_dev.source_id.astype(str)).isdisjoint(set(test.source_id.astype(str)))

protocol={
    'analysis_status':'POST_HOC_SECONDARY_ROBUSTNESS',
    'target_journal':'Signal Processing',
    'detector':'SRNet architecture (Boroumand, Chen, Fridrich 2019)',
    'role':'second independently trained neural steganalyzer',
    'teacher':'SRM-derived local-risk teacher',
    'primary_confirmatory_detector':'enhanced residual CNN from 05e/07',
    'allocator_alpha_frozen':alpha,
    'primary_payload_bpp':primary_bpp,
    'comparison':['joint','predictability'],
    'training_allocation':'random',
    'srnet_train_start':SRNET_TRAIN_START,
    'srnet_train_n':SRNET_TRAIN_N,
    'srnet_dev_start':SRNET_DEV_START,
    'srnet_dev_n':SRNET_DEV_N,
    'epochs_stage1':EPOCHS_STAGE1,
    'epochs_stage2':EPOCHS_STAGE2,
    'micro_pair_batch_size':MICRO_PAIR_BATCH,
    'accumulation_steps':ACCUMULATION_STEPS,
    'effective_pairs_per_optimizer_step':MICRO_PAIR_BATCH*ACCUMULATION_STEPS,
    'lr_stage1':LR_STAGE1,
    'lr_stage2':LR_STAGE2,
    'weight_decay':WEIGHT_DECAY,
    'sanity_auc_threshold':SANITY_AUC_THRESHOLD,
    'fixed_fpr':fixed_fpr,
    'bootstrap_resamples':n_boot,
    'test_results_previously_known':True,
    'test_must_not_be_used_for_detector_fitting_or_selection':True,
    'no_allocator_or_payload_retuning_permitted':True,
    'allocator_sha256':sha256_file(allocator_path),
    'risk_model_sha256':sha256_file(risk_path),
    'test_source_ids_sha256':complete['test_source_ids_sha256'],
    'srnet_train_ids_sha256':stable_id_hash(srnet_train.source_id.astype(str).tolist()),
    'srnet_dev_ids_sha256':stable_id_hash(srnet_dev.source_id.astype(str).tolist()),
}
(OUT/'srnet_protocol_pretest.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
print(json.dumps(protocol,indent=2))


## B. Generate detector-development pairs

Every cover/stego pair is image-disjoint across SRNet train/dev, uses identical payload accounting, and must satisfy exact image and message recovery. These data are detector-development data only.


In [ ]:
def random_order_for_plans(plans,source_id):
    bids=np.asarray([p.block_id for p in plans],dtype=int)
    digest=hashlib.sha256(f'{seed}|srnet|{source_id}'.encode()).digest()
    rng=np.random.default_rng(int.from_bytes(digest[:8],'little'))
    out=bids.copy()
    rng.shuffle(out)
    return out

def make_random_pairs(frame,label):
    covers=[]; stegos=[]; ids=[]; skipped=[]
    for j,row in frame.iterrows():
        sid=str(row.source_id)
        x=read_gray(row.path)
        plans=analyze_blocks(x,bs)
        order=random_order_for_plans(plans,sid)
        rr=run_frozen_image_precomputed(
            x,sid,primary_bpp,'random',{'random':order},[],bs,seed,False,None,plans=plans
        )
        if rr['feasible']:
            if not (rr['exact_image'] and rr['exact_message'] and float(rr['ber'])==0.0):
                raise RuntimeError(f'Reversibility invariant failed for {label}: {sid}')
            covers.append(x); stegos.append(rr['stego']); ids.append(sid)
        else:
            skipped.append(sid)
        if (j+1)%250==0 or j+1==len(frame):
            print(label,j+1,'/',len(frame),'feasible',len(covers))
    return covers,stegos,ids,skipped

train_c,train_s,train_ids,train_skip=make_random_pairs(srnet_train,'srnet-train')
dev_c,dev_s,dev_ids,dev_skip=make_random_pairs(srnet_dev,'srnet-dev')
print('train feasibility:',len(train_c),'/',len(srnet_train))
print('dev feasibility:',len(dev_c),'/',len(srnet_dev))
if len(train_c)<0.95*len(srnet_train) or len(dev_c)<0.95*len(srnet_dev):
    raise RuntimeError('Unexpectedly low detector-development feasibility; inspect before training.')


## C. Train SRNet and apply a pre-test sanity gate

This is the published SRNet topology, but with a fixed project-specific shortened schedule because the original paper used substantially larger training and GPU resources. Validation is diagnostic only: there is no best-epoch selection and no early stopping.

The gate `dev ROC-AUC >= 0.65` is only a detector-validity check. Failure must stop TEST scoring. It does **not** license changes to the frozen RDH allocator. Any revision of the SRNet training protocol must remain entirely inside TRAIN/development data and create a new pre-test protocol lock.


In [ ]:
srnet,history,device=train_srnet(
    train_c,train_s,
    validation=(dev_c,dev_s),
    epochs_stage1=EPOCHS_STAGE1,
    epochs_stage2=EPOCHS_STAGE2,
    micro_pair_batch_size=MICRO_PAIR_BATCH,
    accumulation_steps=ACCUMULATION_STEPS,
    lr_stage1=LR_STAGE1,
    lr_stage2=LR_STAGE2,
    weight_decay=WEIGHT_DECAY,
    seed=seed+10000,
    fixed_fpr=fixed_fpr,
)

hist=pd.DataFrame(history)
hist.to_csv(OUT/'srnet_train_history.csv',index=False)
display(hist)
print('Device:',device)
print('Trainable parameter count:',parameter_count(srnet))

model_path=MODEL_DIR/'srnet_secondary_10.pt'
save_srnet(srnet,model_path,{
    'analysis_status':'post-hoc secondary robustness',
    'architecture':'SRNet topology, Boroumand et al. 2019',
    'primary_payload_bpp':primary_bpp,
    'training_allocation':'random',
    'train_split_only':True,
    'train_pairs':len(train_c),
    'dev_pairs':len(dev_c),
    'epochs_stage1':EPOCHS_STAGE1,
    'epochs_stage2':EPOCHS_STAGE2,
    'micro_pair_batch_size':MICRO_PAIR_BATCH,
    'accumulation_steps':ACCUMULATION_STEPS,
    'lr_stage1':LR_STAGE1,
    'lr_stage2':LR_STAGE2,
    'weight_decay':WEIGHT_DECAY,
    'no_best_epoch_selection':True,
    'seed':seed+10000,
})

dev_cover_scores=score_srnet(srnet,dev_c,device=device,batch_size=4)
dev_stego_scores=score_srnet(srnet,dev_s,device=device,batch_size=4)
y=np.tile([0,1],len(dev_cover_scores))
sc=np.column_stack([dev_cover_scores,dev_stego_scores]).reshape(-1)
met=detector_metrics(y,sc,fixed_fpr)
ci=paired_detector_bootstrap(
    dev_cover_scores,dev_stego_scores,fixed_fpr=fixed_fpr,
    n_resamples=max(n_boot,3000),confidence=confidence,seed=seed+10100,
)
dev_pe=minimal_detection_error(dev_cover_scores,dev_stego_scores)

sanity={
    'sanity_pass':bool(met['auc']>=SANITY_AUC_THRESHOLD),
    'sanity_auc_threshold':SANITY_AUC_THRESHOLD,
    'pairs':len(dev_cover_scores),
    'auc':float(met['auc']),
    'auc_ci_low':float(ci['auc_low']),
    'auc_ci_high':float(ci['auc_high']),
    'tpr_at_5pct_fpr':float(met['tpr_at_fpr']),
    'tpr_ci_low':float(ci['tpr_low']),
    'tpr_ci_high':float(ci['tpr_high']),
    'pe':float(dev_pe),
    'model_sha256':sha256_file(model_path),
    'test_split_scored':False,
}
(OUT/'srnet_dev_sanity.json').write_text(json.dumps(sanity,indent=2),encoding='utf-8')
print(json.dumps(sanity,indent=2))

if not sanity['sanity_pass']:
    raise RuntimeError(
        f"SRNet failed the pre-test sanity gate: AUC={sanity['auc']:.3f} < {SANITY_AUC_THRESHOLD:.2f}. "
        'STOP before TEST scoring. Do not modify the frozen RDH allocator.'
    )
print('SANITY PASSED. Freeze this SRNet before any SRNet TEST scoring.')


## D. Cryptographic-style pre-test lock

The detector checkpoint and protocol are hashed before any SRNet TEST score is computed.


In [ ]:
lock={
    **protocol,
    'srnet_model_sha256':sha256_file(model_path),
    'srnet_model_metadata_sha256':sha256_file(model_path.with_suffix(model_path.suffix+'.json')),
    'dev_sanity_auc':sanity['auc'],
    'dev_sanity_pass':sanity['sanity_pass'],
    'protocol_locked_before_srnet_test_scoring':True,
}
(OUT/'srnet_pretest_lock.json').write_text(json.dumps(lock,indent=2),encoding='utf-8')
print('SRNet pre-test lock SHA256:',sha256_file(OUT/'srnet_pretest_lock.json'))


## E. Frozen TEST scoring at the primary payload only

Only the already specified primary operating point `0.009 net bpp` is evaluated. For every source image we regenerate `predictability` and `joint` stegos from the frozen context, verify exact reversibility, and cross-check net payload and PSNR against notebook 06 before scoring with SRNet.


In [ ]:
srnet,device=load_srnet(model_path)
primary_ids=common[np.isclose(common.target_net_bpp.astype(float),primary_bpp)].source_id.astype(str).tolist()
if len(primary_ids)==0:
    raise RuntimeError('No common-feasible primary-payload IDs.')

test_by_id={str(r.source_id):r for _,r in test.iterrows()}
missing=[sid for sid in primary_ids if sid not in test_by_id]
if missing:
    raise RuntimeError(f'{len(missing)} frozen IDs missing from test manifest')

def load_context(x,sid):
    p=context_dir/f'{sid}.joblib'
    if p.exists():
        ctx=joblib.load(p)
        if ctx.get('context_tag')!=protocol06['context_tag']:
            raise RuntimeError(f'Stale context for {sid}')
        return ctx['orders'],ctx['block_rows'],ctx['plans']
    return prepare_image_context(x,sid,local_risk,alpha,bs,seed)

cover_images=[read_gray(test_by_id[sid].path) for sid in primary_ids]
cover_scores=score_srnet(srnet,cover_images,device=device,batch_size=4)
cover_score_map=dict(zip(primary_ids,map(float,cover_scores)))

def flush_score_batch(model,device,batch_imgs,batch_meta,rows):
    if not batch_imgs:
        return
    ss=score_srnet(model,batch_imgs,device=device,batch_size=4)
    for meta,score in zip(batch_meta,ss):
        cs=cover_score_map[meta['source_id']]
        rows.append({**meta,'cover_score':cs,'stego_score':float(score),'score_delta':float(score-cs)})

rows=[]
for strategy in ['predictability','joint']:
    batch_imgs=[]; batch_meta=[]
    for k,sid in enumerate(primary_ids,1):
        x=read_gray(test_by_id[sid].path)
        orders,br,plans=load_context(x,sid)
        rr=run_frozen_image_precomputed(
            x,sid,primary_bpp,strategy,orders,br,bs,seed,False,None,plans=plans
        )
        if not rr['feasible']:
            raise RuntimeError(f'Frozen common-feasible case became infeasible: {sid} {strategy}')
        if not (rr['exact_image'] and rr['exact_message'] and float(rr['ber'])==0.0):
            raise RuntimeError(f'Reversibility invariant failed: {sid} {strategy}')
        old=per06[
            (per06.source_id==sid) &
            (per06.strategy.astype(str)==strategy) &
            np.isclose(per06.target_net_bpp.astype(float),primary_bpp)
        ]
        if len(old)!=1:
            raise RuntimeError(f'Frozen case lookup failed: {sid} {strategy}')
        old=old.iloc[0]
        if int(rr['net_payload_bits'])!=int(old['net_payload_bits']):
            raise RuntimeError(f'Net-payload mismatch vs frozen 06: {sid} {strategy}')
        if not np.isclose(float(rr['psnr']),float(old['psnr']),rtol=0,atol=1e-10):
            raise RuntimeError(f'PSNR mismatch vs frozen 06: {sid} {strategy}')
        batch_imgs.append(rr['stego'])
        batch_meta.append({'source_id':sid,'strategy':strategy,'target_net_bpp':primary_bpp})
        if len(batch_imgs)>=4:
            flush_score_batch(srnet,device,batch_imgs,batch_meta,rows)
            batch_imgs=[]; batch_meta=[]
        if k%250==0 or k==len(primary_ids):
            print(strategy,k,'/',len(primary_ids))
    flush_score_batch(srnet,device,batch_imgs,batch_meta,rows)

scores=pd.DataFrame(rows)
scores.to_csv(OUT/'srnet_primary_scores.csv',index=False)
print('Saved rows:',len(scores))


## F. Secondary detector metrics and paired method comparison

Report ROC-AUC, `TPR@FPR=5%`, and the classical minimal equal-prior detection error

\[
P_E=\min_\tau\frac{P_{FA}(\tau)+P_{MD}(\tau)}{2}.
\]

For AUC, TPR, and \(P_E\), the comparison is bootstrapped by source image so the two allocation methods remain paired.


In [ ]:
summary_rows=[]
for strategy,g in scores.groupby('strategy',sort=False):
    c0=g.cover_score.to_numpy(float)
    s0=g.stego_score.to_numpy(float)
    y=np.tile([0,1],len(g))
    sc=np.column_stack([c0,s0]).reshape(-1)
    m=detector_metrics(y,sc,fixed_fpr)
    ci=paired_detector_bootstrap(
        c0,s0,fixed_fpr=fixed_fpr,n_resamples=n_boot,
        confidence=confidence,seed=seed+12000+(0 if strategy=='predictability' else 1),
    )
    summary_rows.append({
        'strategy':strategy,'target_net_bpp':primary_bpp,'pairs':len(g),
        'auc':float(m['auc']),'auc_ci_low':float(ci['auc_low']),'auc_ci_high':float(ci['auc_high']),
        'tpr_at_5pct_fpr':float(m['tpr_at_fpr']),
        'tpr_ci_low':float(ci['tpr_low']),'tpr_ci_high':float(ci['tpr_high']),
        'pe':float(minimal_detection_error(c0,s0)),
        'score_delta_mean':float(np.mean(s0-c0)),
    })
summary=pd.DataFrame(summary_rows)
summary.to_csv(OUT/'srnet_primary_summary.csv',index=False)
display(summary)

pred=scores[scores.strategy=='predictability'].set_index('source_id').loc[primary_ids]
joint=scores[scores.strategy=='joint'].set_index('source_id').loc[primary_ids]
c0=joint.cover_score.to_numpy(float)
if not np.allclose(c0,pred.cover_score.to_numpy(float),rtol=0,atol=0):
    raise RuntimeError('Cover-score alignment failure.')

paired=paired_method_bootstrap(
    c0,joint.stego_score.to_numpy(float),pred.stego_score.to_numpy(float),
    fixed_fpr=fixed_fpr,n_resamples=n_boot,confidence=confidence,seed=seed+13000,
)
pe_boot=paired_pe_bootstrap(
    c0,joint.stego_score.to_numpy(float),pred.stego_score.to_numpy(float),
    n_resamples=n_boot,confidence=confidence,seed=seed+13100,
)
paired.update(pe_boot)
paired.update({
    'analysis_status':'POST_HOC_SECONDARY_ROBUSTNESS',
    'method':'joint','reference':'predictability',
    'target_net_bpp':primary_bpp,'alpha':alpha,'pairs':len(primary_ids),
    'srnet_model_sha256':sha256_file(model_path),
    'allocator_sha256':sha256_file(allocator_path),
    'risk_model_sha256':sha256_file(risk_path),
    'no_retuning_permitted':True,
})
(OUT/'srnet_joint_vs_predictability.json').write_text(json.dumps(paired,indent=2),encoding='utf-8')
print(json.dumps(paired,indent=2))


In [ ]:
fig,ax=plt.subplots(figsize=(6.2,4.2))
z=summary.set_index('strategy').loc[['predictability','joint']]
ax.bar(['Predictability','Joint'],z.auc.values)
ax.axhline(0.5,linewidth=1)
ax.set_ylabel('SRNet ROC-AUC')
ax.set_title('Secondary robustness analysis at 0.009 net bpp')
fig.tight_layout()
fig.savefig(OUT/'srnet_primary_auc.png',dpi=300)
fig.savefig(OUT/'srnet_primary_auc.svg')
plt.show()

final={
    'status':'COMPLETE',
    'analysis_status':'POST_HOC_SECONDARY_ROBUSTNESS',
    'primary_payload_bpp':primary_bpp,
    'alpha':alpha,
    'pairs':len(primary_ids),
    'joint_auc':float(z.loc['joint','auc']),
    'predictability_auc':float(z.loc['predictability','auc']),
    'joint_tpr5':float(z.loc['joint','tpr_at_5pct_fpr']),
    'predictability_tpr5':float(z.loc['predictability','tpr_at_5pct_fpr']),
    'joint_pe':float(z.loc['joint','pe']),
    'predictability_pe':float(z.loc['predictability','pe']),
    'auc_diff_joint_minus_predictability':float(paired['auc_diff']),
    'auc_diff_ci_low':float(paired['auc_diff_low']),
    'auc_diff_ci_high':float(paired['auc_diff_high']),
    'tpr_diff_joint_minus_predictability':float(paired['tpr_diff']),
    'tpr_diff_ci_low':float(paired['tpr_diff_low']),
    'tpr_diff_ci_high':float(paired['tpr_diff_high']),
    'pe_diff_joint_minus_predictability':float(paired['pe_diff']),
    'pe_diff_ci_low':float(paired['pe_diff_low']),
    'pe_diff_ci_high':float(paired['pe_diff_high']),
    'interpretation_rule':'supportive if joint has lower AUC/TPR and higher Pe; report the observed result regardless of direction',
    'allocator_retuned':False,
}
(OUT/'srnet_secondary_summary.json').write_text(json.dumps(final,indent=2),encoding='utf-8')
print(json.dumps(final,indent=2))


## Outputs to preserve

After a successful run, preserve `results/srnet_secondary/` and the frozen checkpoint `results/models/srnet_secondary_10.pt`. The four files needed for manuscript integration are `srnet_dev_sanity.json`, `srnet_primary_summary.csv`, `srnet_joint_vs_predictability.json`, and `srnet_secondary_summary.json`.

Do not replace the already frozen primary endpoint with SRNet. In the paper this analysis belongs under **secondary robustness / detector transfer**, and the limitation should state that it was added after inspection of the original confirmatory-CNN test results.
